# 01. Setup e Configuração do Ambiente


In [56]:
# --- Manipulação, Ingestão e Sistema de Arquivos ---
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
# --- Visualização ---
import matplotlib.pyplot as plt
import seaborn as sns
# --- Warnings ---
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [57]:
# --- Configurações de Exibição do Pandas ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# --- Estilo Visual do Matplotlib e Seaborn ---
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
%config InlineBackend.figure_format = 'retina'

print("Ambiente configurado e bibliotecas importadas com sucesso!")

Ambiente configurado e bibliotecas importadas com sucesso!


# 02. Ingestão, Saneamento e Validação Relacional

### Ingestão

In [58]:
# caminho aquivo sql
caminho_sql = Path('..')/'data'/'raw'/'migracao_claro_brasil.sql'

# ler o conteúdo do arquivo como string
with open(caminho_sql, mode='r', encoding='utf-8') as f:
    sql_script = f.read()

# criar conexão SQLite em memória e executar o script
conn = sqlite3.connect(':memory:')
conn.executescript(sql_script)

# carregar cada tabela como DataFrame
df_fato = pd.read_sql('SELECT * FROM FatoMigracao', conn)
df_cliente = pd.read_sql('SELECT * FROM DimCliente', conn)
df_servico = pd.read_sql('SELECT * FROM DimServico', conn)
df_operadora = pd.read_sql('SELECT * FROM DimOperadora', conn)
df_calendario = pd.read_sql('SELECT * FROM dCalendario', conn)

# validação
for nome, df in [('Fato', df_fato), ('Cliente', df_cliente), ('Servico', df_servico),
                ('Operadora', df_operadora), ('Calendario', df_calendario)]:
    print(f'{nome}: {df.shape[0]} linhas x {df.shape[1]} colunas')

Fato: 50000 linhas x 17 colunas
Cliente: 8000 linhas x 6 colunas
Servico: 21 linhas x 4 colunas
Operadora: 7 linhas x 3 colunas
Calendario: 1096 linhas x 5 colunas


### Coerção de tipos

In [59]:
# Conversão de colunas de data para datetime
df_fato['id_data'] = pd.to_datetime(df_fato['id_data'])
df_calendario['Data'] = pd.to_datetime(df_calendario['Data'])

# Conversão de colunas categóricas de baixa cardinalidade
cols_categoricas = [
    'direcao', 'categoria_servico', 'status_portabilidade',
    'uf', 'regiao', 'motivo', 'canal',
    'operadora_origem', 'operadora_destino'
]

for col in cols_categoricas:
    df_fato[col] = df_fato[col].astype('category')

### Nulos e duplicatas

In [60]:
dfs = {
    'Fato': df_fato,
    'Cliente': df_cliente,
    'Servico': df_servico,
    'Operadora': df_operadora,
    'Calendario': df_calendario
}

for nome, df in dfs.items():
    nulos = df.isnull().sum().sum()
    duplicatas = df.duplicated().sum()
    print(f'{nome}: {nulos} nulos | {duplicatas} duplicatas')

Fato: 0 nulos | 0 duplicatas
Cliente: 0 nulos | 0 duplicatas
Servico: 0 nulos | 0 duplicatas
Operadora: 0 nulos | 0 duplicatas
Calendario: 0 nulos | 0 duplicatas


**Resultado:** Nenhum nulo ou duplicata encontrado nas 5 tabelas. 
As anomalias injetadas pelo gerador provavelmente se manifestam como 
outliers em variáveis numéricas ou inconsistências categóricas.

### Renomeação de colunas

In [ ]:
df_fato = df_fato.rename(columns={
    'direcao': 'direcao_migracao',
    'motivo': 'motivo_migracao',
    'canal': 'canal_aquisicao'
})

### Auditoria de colunas desnormalizadas

In [ ]:
# Merge: geolocalização da Fato vs. DimCliente
df_audit_geo = df_fato[['id_migracao', 'id_cliente', 'uf', 'regiao']].merge(
    df_cliente[['id_cliente', 'uf', 'regiao']],
    on='id_cliente',
    how='left',
    suffixes=('_fato', '_cliente')
)

# Contagem de divergências
divergencias_uf = (df_audit_geo['uf_fato'] != df_audit_geo['uf_cliente']).sum()
divergencias_regiao = (df_audit_geo['regiao_fato'] != df_audit_geo['regiao_cliente']).sum()
pct_uf = (divergencias_uf / len(df_audit_geo)) * 100
pct_regiao = (divergencias_regiao / len(df_audit_geo)) * 100

print("=== AUDITORIA: GEOLOCALIZAÇÃO (Fato vs. DimCliente) ===")
print(f"Total de registros : {len(df_audit_geo):,}")
print(f"Divergências de UF : {divergencias_uf:,} ({pct_uf:.2f}%)")
print(f"Divergências de Região : {divergencias_regiao:,} ({pct_regiao:.2f}%)")

# Amostra de divergências (se existirem)
df_divergencias = df_audit_geo[
    (df_audit_geo['uf_fato'] != df_audit_geo['uf_cliente']) |
    (df_audit_geo['regiao_fato'] != df_audit_geo['regiao_cliente'])
]

if not df_divergencias.empty:
    print(f"\nAmostra das divergências ({len(df_divergencias)} encontradas):")
    display(df_divergencias.head())
else:
    print("\n[OK] Nenhuma divergência — colunas desnormalizadas validadas.")

=== AUDITORIA: GEOLOCALIZAÇÃO (Fato vs. DimCliente) ===
Total de registros           : 50,000
Divergências de UF           : 0 (0.00%)
Divergências de Região       : 0 (0.00%)

[OK] Nenhuma divergência — colunas desnormalizadas validadas.


**Auditoria geolocalização (Fato vs. DimCliente):** 0 divergências em UF e Região 
nos 50.000 registros. Colunas desnormalizadas na Fato são confiáveis para uso 
direto nas análises. A decisão de não fazer merge obrigatório com DimCliente 
para cortes geográficos está validada.